# Tugas Besar 2 IF3170 Inteligensi Artifisial

*Klasifikasi pembatalan pemesanan hotel (`is_canceled`).*

## IF3170_G08_Teazzi

- 13524034 - Jason Edward Salim
- 13524064 - Stefani Angeline Oroh
- 13524067 - Bryan Pratama Putra Hendra
- 13524079 - Angelina Andra Alanna
- 13524106 - Michael James Liman

## Table of Contents (CRISP-DM)

1. [**Business Understanding**](#1)
2. [**Data Understanding**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling**](#4)
5. [**Evaluation**](#5)
6. [**Deployment**](#6)

## Initialization

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"))
from config import Settings, seed_everything
from eda import types, missing, duplicates, pie, bar, histogram, boxplot, iqr_outliers, target_rate, correlation
from preprocessing import load_data, split, Preprocessor, Pipeline
from validation import fit, holdout, kfold, compare, tune, best, importance, confusion, report, misclassified, submit
from models import KNN, LogisticRegression, SVM, DTL, BayesNetwork
seed_everything()

In [ ]:
full_train, test = load_data()
X_train, X_val, y_train, y_val = split(full_train)
eda_train = full_train.loc[X_train.index]
pd.Series({"train": len(full_train), "train_fit": len(X_train), "validation": len(X_val), "test": len(test)})

---

# Business Understanding <a name="1"></a>

---

Memprediksi apakah sebuah pemesanan hotel akan dibatalkan (`is_canceled = 1`) atau tidak (`is_canceled = 0`). Metrik evaluasi: macro F1-score.

> TODO: konteks bisnis dan dampak false positive / false negative.

---

# Data Understanding <a name="2"></a>

---

## Data Overview

In [ ]:
eda_train

In [ ]:
display(types(eda_train))
display(types(test))

In [ ]:
missing(eda_train, test)

In [ ]:
duplicates(eda_train, test)

## Target Distribution

In [ ]:
pie(eda_train)
bar(eda_train)

## Feature Analysis

In [ ]:
num_col = eda_train.select_dtypes("number").columns.drop(Settings.TARGET)
cat_col = eda_train.columns.difference(num_col).drop([Settings.ID, Settings.TARGET])
display(eda_train[num_col].describe().T)
display(eda_train[cat_col].describe().T)

In [ ]:
histogram(eda_train, num_col)
boxplot(eda_train, num_col)

In [ ]:
iqr_outliers(eda_train, num_col)

In [ ]:
target_rate(eda_train, list(cat_col) + ['special_requests', 'arrival_month', 'is_returning_guest'])

In [ ]:
correlation(eda_train)

#### Insights

> TODO

---

# Data Preparation <a name="3"></a>

---

Split stratified memakai seed validasi terpisah dan dilakukan sebelum EDA berbasis target. EDA menggunakan bagian training saja. Di setiap fold, preprocessor di-fit hanya pada training fold; untuk submission, preprocessor di-fit ulang pada seluruh data train. Profil fitur berulang dipertahankan karena ID booking berbeda.


In [ ]:
X_test = test.drop(columns=[Settings.ID])
pd.Series({"X_train": len(X_train), "X_val": len(X_val), "X_test": len(X_test)})

In [ ]:
def make_preprocessor():
    return Preprocessor(
        zero_as_missing=[],
        num_strategy="median",
        cap=["days_before_arrival", "nightly_rate_idr"],
        ordinal={"meal_package": ["Room Only", "Breakfast", "Half Board", "Full Board"]},
        scaler="standard",
    )
preprocessor_factory = make_preprocessor

#### Keputusan

Nilai numerik yang hilang diimputasi dengan median dan kategori dengan modus. Indikator harga nol dibuat sebelum clipping agar nilai nol tidak tersamarkan oleh batas bawah harga. Tipe kamar diperlakukan nominal melalui one-hot encoding; paket makan tetap ordinal karena urutannya menunjukkan tambahan layanan. Fitur biner dan one-hot dibiarkan pada skala 0–1, sedangkan fitur numerik lain distandardisasi. Statistik preprocessing di-fit di dalam training fold agar validasi tidak ikut membentuk transformasi.


---

# Modeling <a name="4"></a>

---

Model from scratch ada di `src/models/` (implementasi `fit`/`predict`). Model sklearn dipakai sebagai pembanding. `METHOD` menentukan penanganan class imbalance (`None`, `'smote'`, `'undersample'`), hanya diterapkan pada data latih.

In [ ]:
METHOD = None
# Pipeline(model, select=k, pca=n) adds per-model feature selection and PCA
models = {
    "knn": KNN(k=5),
    "lr": LogisticRegression(lr=0.01, n_iter=1000),
    "svm": SVM(kernel="rbf", C=3, gamma=0.1),
    "dtl": DTL(max_depth=None),
    "bn": BayesNetwork(),
}

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
sk_models = {
    "sk_knn": KNeighborsClassifier(5),
    "sk_lr": SkLogisticRegression(max_iter=1000),
    "sk_svm": SVC(kernel="rbf", C=3, gamma=0.1),
    "sk_dtl": DecisionTreeClassifier(max_depth=8, random_state=Settings.SEED),
    "sk_nb": GaussianNB(),
}

## SVM Feature Selection

SVM diselesaikan dalam bentuk dual dengan SMO, sehingga untuk kernel linear w = Σ αᵢyᵢxᵢ. Fitur diranking berdasarkan |w| SVM linear yang dilatih pada seluruh fitur, lalu `select` fitur teratas dipakai untuk melatih ulang SVM. Model final memakai kernel RBF, yang tidak memiliki w eksplisit.

In [ ]:
prep_display = preprocessor_factory()
X_train_display = prep_display.fit_transform(X_train, y_train)
svm_fitted = fit(Pipeline(SVM(kernel="linear", C=0.1), select=10), X_train_display, y_train)
pd.Series(X_train_display.columns[svm_fitted.keep], name="svm selected features")

---

# Evaluation <a name="5"></a>

---

## Model Comparison

Holdout memakai seed terpisah dari eksperimen sebelumnya; K-Fold pada `X_train` dipakai untuk memilih model dan hyperparameter. Karena analisis eksplorasi sebelumnya sempat melihat target seluruh data train, skor holdout tetap diagnostik dan bukan estimasi generalisasi yang sepenuhnya independen.

In [ ]:
all_models = {**models, **sk_models}
compare(
    holdout(all_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
    kfold(all_models, X_train, y_train, METHOD, preprocessor_factory),
)

## Hyperparameter Tuning

In [ ]:
grids = {
    "knn": {"k": [5, 15, 31]},
    "lr": {"lr": [0.01, 0.1], "n_iter": [1000]},
    "svm": {"kernel": ["rbf"], "C": [1, 3, 5], "gamma": [0.1]},
    "dtl": {"max_depth": [5, 8, None]},
    "bn": {"n_bins": [3, 5, 10]},
}
grid_tables = {
    name: tune(models[name], grid, X_train, y_train, METHOD, preprocessor_factory)
    for name, grid in grids.items()
}
tuned_models = {name: best(models[name], table) for name, table in grid_tables.items()}
grid_tables["knn"].head()

In [ ]:
ranking = compare(
    holdout(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
    kfold(tuned_models, X_train, y_train, METHOD, preprocessor_factory),
)
ranking

## Error Analysis

In [ ]:
# Pakai baseline scikit-learn jika belum ada model from scratch yang bisa dijalankan.
if ranking.empty:
    ranking = compare(
        holdout(sk_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory),
        kfold(sk_models, X_train, y_train, METHOD, preprocessor_factory),
    )
    tuned_models = {**tuned_models, **sk_models}
best_name = ranking.index[0]
best_model = tuned_models[best_name]
best_name, best_model

In [ ]:
confusion(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory)

In [ ]:
report(tuned_models, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory)

In [ ]:
wrong_rows = misclassified(
    best_model, full_train, X_train, y_train, X_val, y_val, METHOD, preprocessor_factory
)
print(f"{len(wrong_rows)} dari {len(X_val)} baris salah")
wrong_rows["predicted"].map({1: "false positive", 0: "false negative"}).value_counts()

In [ ]:
importance(sk_models["sk_dtl"], X_train_display, y_train).head(10)

#### Insights

> TODO

---

# Deployment <a name="6"></a>

---

Model terbaik dilatih ulang dengan seluruh data train mentah. Preprocessor di-fit pada seluruh train, dipakai untuk mentransformasi train dan test, lalu disimpan bersama artefak model.


In [ ]:
X_full = full_train.drop(columns=[Settings.ID, Settings.TARGET])
y_full = full_train[Settings.TARGET]
prep_final = preprocessor_factory()
X_full_prepared = prep_final.fit_transform(X_full, y_full)
X_test_prepared = prep_final.transform(test.drop(columns=[Settings.ID]))
submit(
    best_model,
    best_name,
    X_full_prepared,
    y_full,
    X_test_prepared,
    test[Settings.ID],
    METHOD,
    preprocessor=prep_final,
)